# EMG Datenverarbeitungspipeline
### nach Ellenberger et al. (2021)

**Schritte:**
1. Daten laden & Resampling (2100 Hz → 2000 Hz für S01–S04)
2. Rectifizierung
3. Bandpassfilterung (20–500 Hz, 4. Ordnung Zero-Lag Butterworth)
4. RMS-Glättung (symmetrisches gleitendes 30 ms Fenster)
5. Zeitnormalisierung auf 101 Datenpunkte

> Visualisierungen nach jedem Schritt zur Qualitätskontrolle

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import butter, sosfiltfilt, resample_poly
from scipy.interpolate import interp1d
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Farbpalette pro Muskel
MUSCLE_COLORS = {
    'Biceps Femoris':       '#e63946',
    'Gastrocnemius medial': '#457b9d',
    'Gluteus Medius':       '#2a9d8f',
    'Semitendinosus':       '#e9c46a',
    'Vastus Lateralis':     '#f4a261',
}

print('✅ Imports erfolgreich')

## 0 · Konfiguration & Datenpfade

In [ ]:
# ─── Konfiguration ──────────────────────────────────────────────────────────────

# Pfad zum Datenordner  →  bitte anpassen!
DATA_DIR = Path(r'C:\Users\Greta\OneDrive\Desktop\MCI\3-SS2026\BA\BA_Daten_EMG\data\preprocessed_emg_data\S01\01_PER\CMJ')

# Sampling-Raten
FS_ORIGINAL = 2100   # S01–S04
FS_TARGET   = 2000   # Ziel (und native Rate S05–S11)

# Bandpassfilter
BP_LOW  = 20    # Hz
BP_HIGH = 500   # Hz
FILTER_ORDER = 4

# RMS-Fenster
RMS_WINDOW_MS = 30   # ms

# Zeitnormalisierung
N_NORM_POINTS = 101

# Muskelspalten (ohne L_ / R_ Prefix)
MUSCLE_NAMES = [
    'Biceps Femoris',
    'Gastrocnemius medial',
    'Gluteus Medius',
    'Semitendinosus',
    'Vastus Lateralis',
]

# Welche Datei soll für die Visualisierungen verwendet werden?
VIZ_FILE = DATA_DIR / 'CMJ_01.csv'   # ← erste Wiederholung
VIZ_SIDE = 'L'                        # 'L' oder 'R'

print('✅ Konfiguration gesetzt')
print(f'   Datenpfad : {DATA_DIR}')
print(f'   Ziel-fs   : {FS_TARGET} Hz')
print(f'   RMS-Fenster: {RMS_WINDOW_MS} ms')

---
## SCHRITT 1 · Daten laden & Resampling

S01–S04 wurden mit **2100 Hz** aufgenommen → wird auf **2000 Hz** resamplt (Faktor 20/21).  
Für S05–S11 (2000 Hz) wird dieser Schritt übersprungen.

In [ ]:
def load_and_resample(filepath: Path,
                       fs_orig: int = FS_ORIGINAL,
                       fs_target: int = FS_TARGET) -> tuple[pd.DataFrame, int]:
    """
    Lädt eine CSV-Datei und resampelt die EMG-Spalten auf fs_target.
    Gibt (DataFrame, tatsächliche fs nach Resampling) zurück.
    """
    df = pd.read_csv(filepath)

    # Tatsächliche Sampling-Rate aus den Zeitstempeln schätzen
    dt_median = df['time_s'].diff().median()
    fs_detected = round(1.0 / dt_median)

    emg_cols = [c for c in df.columns
                if any(m in c for m in MUSCLE_NAMES)]

    if fs_detected != fs_target:
        # Polyphasen-Resampling: up=20, down=21  (2000/2100 = 20/21)
        from math import gcd
        g   = gcd(fs_target, fs_detected)
        up  = fs_target   // g
        down= fs_detected // g

        resampled = {}
        for col in emg_cols:
            resampled[col] = resample_poly(df[col].values, up, down)

        n_new   = len(next(iter(resampled.values())))
        t_new   = np.linspace(df['time_s'].iloc[0],
                               df['time_s'].iloc[-1],
                               n_new)

        df_out = pd.DataFrame({'time_s': t_new})
        # Meta-Spalten übernehmen (erste Zeile, nicht-EMG)
        meta_cols = [c for c in df.columns
                     if c != 'time_s' and c not in emg_cols]
        for mc in meta_cols:
            df_out[mc] = df[mc].iloc[0]   # skalare Metadaten
        for col, arr in resampled.items():
            df_out[col] = arr

        print(f'  Resampling: {fs_detected} Hz → {fs_target} Hz '
              f'({len(df)} → {n_new} Samples)')
        return df_out, fs_target
    else:
        print(f'  Keine Resampling nötig (fs = {fs_detected} Hz)')
        return df, fs_detected


# ── Demo: Viz-Datei laden ──────────────────────────────────────────────────────
print(f'Lade: {VIZ_FILE.name}')
df_raw, fs = load_and_resample(VIZ_FILE)
print(f'  Samples nach Laden: {len(df_raw)} | fs = {fs} Hz')

In [ ]:
# ── Plot: Rohsignal nach Resampling ────────────────────────────────────────────
emg_cols_side = [f'{VIZ_SIDE}_{m}' for m in MUSCLE_NAMES]
t = df_raw['time_s'].values

fig, axes = plt.subplots(len(MUSCLE_NAMES), 1,
                          figsize=(13, 10), sharex=True)
fig.suptitle(f'SCHRITT 1 · Rohsignal nach Resampling\n'
             f'{VIZ_FILE.name} – Seite {VIZ_SIDE}  (fs = {fs} Hz)',
             fontsize=14, fontweight='bold')

for ax, muscle in zip(axes, MUSCLE_NAMES):
    col = f'{VIZ_SIDE}_{muscle}'
    ax.plot(t, df_raw[col].values,
            color=MUSCLE_COLORS[muscle], lw=0.7, alpha=0.9)
    ax.set_ylabel('Amplitude\n[µV]', fontsize=9)
    ax.set_title(muscle, loc='left', fontsize=10, pad=2)

axes[-1].set_xlabel('Zeit [s]')
plt.tight_layout()
plt.show()

---
## SCHRITT 2 · Rectifizierung

Das EMG-Rohsignal ist biphasisch (positive und negative Ausschläge).  
→ **Gleichrichtung** durch Absolutwertbildung: `|signal|`

In [ ]:
def rectify(df: pd.DataFrame) -> pd.DataFrame:
    """Gleichrichtet alle EMG-Spalten (Absolutwert)."""
    df_out = df.copy()
    emg_cols = [c for c in df.columns
                if any(m in c for m in MUSCLE_NAMES)]
    for col in emg_cols:
        df_out[col] = np.abs(df_out[col].values)
    return df_out


df_rect = rectify(df_raw)
print('✅ Rectifizierung abgeschlossen')

In [ ]:
# ── Plot: Rohsignal vs. rectifiziertes Signal (ein Muskel, Zoom) ───────────────
muscle_demo = MUSCLE_NAMES[0]   # Biceps Femoris als Beispiel
col_demo    = f'{VIZ_SIDE}_{muscle_demo}'

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.suptitle(f'SCHRITT 2 · Rectifizierung – {muscle_demo}\n'
             f'{VIZ_FILE.name}', fontsize=14, fontweight='bold')

ax1.plot(t, df_raw[col_demo].values,
         color='steelblue', lw=0.8, label='Rohsignal')
ax1.axhline(0, color='black', lw=0.5, ls='--')
ax1.set_ylabel('Amplitude [µV]')
ax1.set_title('Rohsignal (biphasisch)', loc='left', fontsize=10)
ax1.legend(fontsize=9)

ax2.plot(t, df_rect[col_demo].values,
         color=MUSCLE_COLORS[muscle_demo], lw=0.8, label='Rectifiziert |signal|')
ax2.set_ylabel('Amplitude [µV]')
ax2.set_xlabel('Zeit [s]')
ax2.set_title('Gleichgerichtetes Signal', loc='left', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## SCHRITT 3 · Bandpassfilterung

- **4. Ordnung Zero-Lag Butterworth** (`sosfiltfilt` = forwards + backwards → kein Phasenverzug)
- **Bandpass: 20 – 500 Hz**

> *Hinweis:* Die Filterung wird auf das **rectifizierte** Signal angewendet (Reihenfolge: rectify → filter, wie in Ellenberger et al.).

In [ ]:
def bandpass_filter(df: pd.DataFrame,
                     fs: int     = FS_TARGET,
                     low: float  = BP_LOW,
                     high: float = BP_HIGH,
                     order: int  = FILTER_ORDER) -> pd.DataFrame:
    """
    Wendet einen 4. Ordnung Zero-Lag Butterworth Bandpassfilter
    auf alle EMG-Spalten an.
    """
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq],
                 btype='bandpass', output='sos')

    df_out = df.copy()
    emg_cols = [c for c in df.columns
                if any(m in c for m in MUSCLE_NAMES)]
    for col in emg_cols:
        df_out[col] = sosfiltfilt(sos, df_out[col].values)
    return df_out


df_filt = bandpass_filter(df_rect, fs=fs)
print(f'✅ Bandpassfilterung abgeschlossen  '
      f'({BP_LOW}–{BP_HIGH} Hz, {FILTER_ORDER}. Ordnung Zero-Lag Butterworth)')

In [ ]:
# ── Plot: vor vs. nach Filterung (alle Muskeln einer Seite) ───────────────────
fig, axes = plt.subplots(len(MUSCLE_NAMES), 1,
                          figsize=(13, 10), sharex=True)
fig.suptitle(f'SCHRITT 3 · Bandpassfilterung ({BP_LOW}–{BP_HIGH} Hz)\n'
             f'{VIZ_FILE.name} – Seite {VIZ_SIDE}',
             fontsize=14, fontweight='bold')

for ax, muscle in zip(axes, MUSCLE_NAMES):
    col = f'{VIZ_SIDE}_{muscle}'
    ax.plot(t, df_rect[col].values,
            color='lightgray', lw=0.7, label='rectifiziert', zorder=1)
    ax.plot(t, df_filt[col].values,
            color=MUSCLE_COLORS[muscle], lw=1.2, label='gefiltert', zorder=2)
    ax.set_ylabel('µV', fontsize=9)
    ax.set_title(muscle, loc='left', fontsize=10, pad=2)
    if muscle == MUSCLE_NAMES[0]:
        ax.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Zeit [s]')
plt.tight_layout()
plt.show()

In [ ]:
# ── Zusatz-Plot: Frequenzspektrum vor/nach Filter (ein Muskel) ────────────────
from scipy.fft import rfft, rfftfreq

col_demo = f'{VIZ_SIDE}_{muscle_demo}'
sig_raw  = df_rect[col_demo].values
sig_filt = df_filt[col_demo].values
N        = len(sig_raw)
freqs    = rfftfreq(N, d=1.0/fs)

psd_raw  = np.abs(rfft(sig_raw))  / N
psd_filt = np.abs(rfft(sig_filt)) / N

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd_raw,  color='lightgray', lw=0.8,
            label='rectifiziert', zorder=1)
ax.semilogy(freqs, psd_filt, color=MUSCLE_COLORS[muscle_demo],
            lw=1.2, label='gefiltert', zorder=2)
ax.axvspan(0,       BP_LOW,  alpha=0.15, color='red',   label='gestoppt')
ax.axvspan(BP_HIGH, fs/2,    alpha=0.15, color='red')
ax.axvline(BP_LOW,  color='red', ls='--', lw=1)
ax.axvline(BP_HIGH, color='red', ls='--', lw=1, label=f'{BP_LOW}/{BP_HIGH} Hz Grenzfreq.')
ax.set_xlim(0, fs/2)
ax.set_xlabel('Frequenz [Hz]')
ax.set_ylabel('Amplitude (log)')
ax.set_title(f'Frequenzspektrum – {muscle_demo}', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## SCHRITT 4 · RMS-Glättung

**Symmetrisches gleitendes 30 ms RMS-Fenster** (point-by-point).

- Fensterlänge = `round(0.030 × fs)` Samples (ungerade → symmetrisch)
- An den Rändern wird mit dem nächsten verfügbaren Sample aufgefüllt (`mode='reflect'`)

> RMS = √( mean( x² ) ) über das Fenster

In [ ]:
def rms_smooth(signal: np.ndarray, fs: int,
               window_ms: float = RMS_WINDOW_MS) -> np.ndarray:
    """
    Symmetrisches gleitendes RMS mit Fensterlänge window_ms [ms].
    """
    win_samples = int(round(window_ms / 1000.0 * fs))
    if win_samples % 2 == 0:
        win_samples += 1          # ungerade für symmetrisches Fenster
    half = win_samples // 2

    # Signal am Rand spiegeln, damit Kantenpunkte korrekt behandelt werden
    padded = np.pad(signal, half, mode='reflect')
    sq     = padded ** 2

    # Kumulierte Summe für schnelles gleitendes Mittel
    cs     = np.cumsum(sq)
    cs     = np.concatenate([[0], cs])
    rms    = np.sqrt((cs[win_samples:] - cs[:-win_samples]) / win_samples)
    return rms


def apply_rms_smooth(df: pd.DataFrame, fs: int,
                      window_ms: float = RMS_WINDOW_MS) -> pd.DataFrame:
    df_out   = df.copy()
    emg_cols = [c for c in df.columns
                if any(m in c for m in MUSCLE_NAMES)]
    win_samples = int(round(window_ms / 1000.0 * fs))
    if win_samples % 2 == 0:
        win_samples += 1
    for col in emg_cols:
        df_out[col] = rms_smooth(df_out[col].values, fs, window_ms)
    return df_out


win_samples = int(round(RMS_WINDOW_MS / 1000.0 * fs))
if win_samples % 2 == 0: win_samples += 1

df_rms = apply_rms_smooth(df_filt, fs=fs)
print(f'✅ RMS-Glättung abgeschlossen')
print(f'   Fensterlänge: {win_samples} Samples = {win_samples/fs*1000:.1f} ms @ {fs} Hz')

In [ ]:
# ── Plot: gefiltert vs. RMS-geglättet (alle Muskeln) ─────────────────────────
fig, axes = plt.subplots(len(MUSCLE_NAMES), 1,
                          figsize=(13, 10), sharex=True)
fig.suptitle(f'SCHRITT 4 · RMS-Glättung ({RMS_WINDOW_MS} ms)\n'
             f'{VIZ_FILE.name} – Seite {VIZ_SIDE}',
             fontsize=14, fontweight='bold')

for ax, muscle in zip(axes, MUSCLE_NAMES):
    col = f'{VIZ_SIDE}_{muscle}'
    ax.fill_between(t, df_filt[col].values,
                    color=MUSCLE_COLORS[muscle], alpha=0.3,
                    label='gefiltert', zorder=1)
    ax.plot(t, df_rms[col].values,
            color=MUSCLE_COLORS[muscle], lw=1.8,
            label='RMS', zorder=2)
    ax.set_ylabel('µV', fontsize=9)
    ax.set_title(muscle, loc='left', fontsize=10, pad=2)
    if muscle == MUSCLE_NAMES[0]:
        ax.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Zeit [s]')
plt.tight_layout()
plt.show()

---
## SCHRITT 5 · Zeitnormalisierung auf 101 Datenpunkte (0–100 %)

Das Signal wird mittels **kubischer Interpolation** auf genau 101 Punkte resampelt,  
die dem Bewegungsintervall von 0 % bis 100 % entsprechen.

In [ ]:
def time_normalize(df: pd.DataFrame,
                    n_points: int = N_NORM_POINTS) -> pd.DataFrame:
    """
    Normalisiert alle EMG-Spalten auf n_points (0–100 %) mittels
    kubischer Interpolation.
    """
    t_orig   = df['time_s'].values
    t_norm   = np.linspace(t_orig[0], t_orig[-1], n_points)
    pct_axis = np.linspace(0, 100, n_points)

    emg_cols = [c for c in df.columns
                if any(m in c for m in MUSCLE_NAMES)]

    out = {'pct': pct_axis}
    for col in emg_cols:
        f_interp = interp1d(t_orig, df[col].values,
                            kind='cubic', fill_value='extrapolate')
        out[col]  = f_interp(t_norm)

    return pd.DataFrame(out)


df_norm = time_normalize(df_rms)
pct     = df_norm['pct'].values

print(f'✅ Zeitnormalisierung abgeschlossen')
print(f'   Samples: {len(df_rms)} → {len(df_norm)} Punkte (0–100 %)')

In [ ]:
# ── Plot: zeitnormalisiertes Signal ──────────────────────────────────────────
fig, axes = plt.subplots(len(MUSCLE_NAMES), 1,
                          figsize=(13, 10), sharex=True)
fig.suptitle(f'SCHRITT 5 · Zeitnormalisierung (0–100 %, n={N_NORM_POINTS})\n'
             f'{VIZ_FILE.name} – Seite {VIZ_SIDE}',
             fontsize=14, fontweight='bold')

for ax, muscle in zip(axes, MUSCLE_NAMES):
    col = f'{VIZ_SIDE}_{muscle}'
    ax.plot(pct, df_norm[col].values,
            color=MUSCLE_COLORS[muscle], lw=2)
    ax.scatter(pct, df_norm[col].values,
               color=MUSCLE_COLORS[muscle], s=15, zorder=3, alpha=0.6)
    ax.set_ylabel('µV', fontsize=9)
    ax.set_title(muscle, loc='left', fontsize=10, pad=2)

axes[-1].set_xlabel('Bewegungszyklus [%]')
plt.tight_layout()
plt.show()

---
## Gesamtpipeline · Zusammenfassung in einem Plot

Alle Verarbeitungsschritte für einen Muskel im Überblick.

In [ ]:
# ── Übersichtsplot: alle 5 Schritte für einen Muskel ─────────────────────────
muscle_demo = MUSCLE_NAMES[4]   # Vastus Lateralis
col_demo    = f'{VIZ_SIDE}_{muscle_demo}'
color       = MUSCLE_COLORS[muscle_demo]

steps = [
    ('1 · Roh (resampelt)',    t,   df_raw[col_demo].values,   'steelblue'),
    ('2 · Rectifiziert',       t,   df_rect[col_demo].values,  'slategray'),
    ('3 · Bandpassgefiltert',  t,   df_filt[col_demo].values,  color),
    ('4 · RMS-geglättet',      t,   df_rms[col_demo].values,   color),
    ('5 · Zeitnormalisiert',   pct, df_norm[col_demo].values,  color),
]

fig, axes = plt.subplots(5, 1, figsize=(14, 14))
fig.suptitle(f'Pipeline-Übersicht – {muscle_demo}\n{VIZ_FILE.name}',
             fontsize=15, fontweight='bold')

xlabels = ['Zeit [s]'] * 4 + ['Bewegungszyklus [%]']

for ax, (title, x, y, c), xl in zip(axes, steps, xlabels):
    ax.plot(x, y, color=c, lw=1.2 if xl != 'Bewegungszyklus [%]' else 2)
    ax.set_title(title, loc='left', fontsize=11, fontweight='bold', pad=3)
    ax.set_ylabel('µV', fontsize=9)
    ax.set_xlabel(xl, fontsize=9)
    if '5' in title:
        ax.scatter(x, y, s=12, color=c, zorder=3, alpha=0.7)
        ax.set_xlim(-1, 101)

plt.tight_layout()
plt.show()

---
## Batch-Verarbeitung · Alle Dateien einer Session

Die komplette Pipeline wird auf **alle 9 Dateien** der Session angewendet.  
Am Ende werden Mittelwert ± SD pro Muskel über die 3 Wiederholungen dargestellt.

In [ ]:
def process_file(filepath: Path,
                  fs_orig: int   = FS_ORIGINAL,
                  fs_target: int = FS_TARGET) -> tuple[pd.DataFrame, int]:
    """
    Wendet die komplette EMG-Pipeline auf eine Datei an.
    Gibt (normalisierter DataFrame, fs) zurück.
    """
    df, fs = load_and_resample(filepath, fs_orig, fs_target)
    df     = rectify(df)
    df     = bandpass_filter(df, fs=fs)
    df     = apply_rms_smooth(df, fs=fs)
    df     = time_normalize(df)
    return df, fs


# ── Alle CSV-Dateien im Ordner verarbeiten ─────────────────────────────────────
csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Gefundene Dateien: {len(csv_files)}')
for f in csv_files:
    print(f'  {f.name}')

results = {}   # filename → normalized DataFrame

for filepath in csv_files:
    print(f'\n→ Verarbeite: {filepath.name}')
    try:
        df_processed, _ = process_file(filepath)
        results[filepath.name] = df_processed
        print(f'   ✅ OK  ({len(df_processed)} Punkte)')
    except Exception as e:
        print(f'   ❌ Fehler: {e}')

print(f'\n✅ Batch-Verarbeitung abgeschlossen ({len(results)}/{len(csv_files)} Dateien)')

In [ ]:
# ── Plot: Mittelwert ± SD über alle Wiederholungen (CMJ bilateral) ─────────────
# Nur CMJ_01/02/03 für bilateral, links, rechts getrennt

def plot_mean_sd(file_keys: list, side: str, title_prefix: str):
    available = [k for k in file_keys if k in results]
    if not available:
        print(f'Keine Dateien für {title_prefix} gefunden.')
        return

    fig, axes = plt.subplots(len(MUSCLE_NAMES), 1,
                              figsize=(13, 10), sharex=True)
    fig.suptitle(f'{title_prefix} – Seite {side}\nMittelwert ± SD ({len(available)} Trials)',
                 fontsize=14, fontweight='bold')

    pct_ax = np.linspace(0, 100, N_NORM_POINTS)

    for ax, muscle in zip(axes, MUSCLE_NAMES):
        col   = f'{side}_{muscle}'
        color = MUSCLE_COLORS[muscle]

        all_trials = []
        for key in available:
            df_t = results[key]
            if col in df_t.columns:
                all_trials.append(df_t[col].values)

        if not all_trials:
            ax.set_title(f'{muscle} – keine Daten', loc='left')
            continue

        arr  = np.array(all_trials)   # shape: (n_trials, 101)
        mean = arr.mean(axis=0)
        sd   = arr.std(axis=0)

        for i, trial in enumerate(arr):
            ax.plot(pct_ax, trial, color=color, lw=0.7, alpha=0.3,
                    label='Einzeltrials' if i == 0 else None)

        ax.fill_between(pct_ax, mean - sd, mean + sd,
                        color=color, alpha=0.3, label='±1 SD')
        ax.plot(pct_ax, mean, color=color, lw=2.5, label='Mittelwert')

        ax.set_ylabel('µV', fontsize=9)
        ax.set_title(muscle, loc='left', fontsize=10, pad=2)
        if muscle == MUSCLE_NAMES[0]:
            ax.legend(fontsize=8, loc='upper right', ncol=3)

    axes[-1].set_xlabel('Bewegungszyklus [%]')
    plt.tight_layout()
    plt.show()


# Bilateral CMJ
plot_mean_sd(['CMJ_01.csv', 'CMJ_02.csv', 'CMJ_03.csv'],
             side='L', title_prefix='CMJ bilateral')

# Unilateral Links
plot_mean_sd(['CMJ_L_01.csv', 'CMJ_L_02.csv', 'CMJ_L_03.csv'],
             side='L', title_prefix='CMJ unilateral Links')

# Unilateral Rechts
plot_mean_sd(['CMJ_R_01.csv', 'CMJ_R_02.csv', 'CMJ_R_03.csv'],
             side='R', title_prefix='CMJ unilateral Rechts')

---
## Export · Verarbeitete Daten speichern

In [ ]:
# ── Alle verarbeiteten Dateien als CSV speichern ───────────────────────────────
OUTPUT_DIR = DATA_DIR.parent / 'processed_emg'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, df_proc in results.items():
    out_path = OUTPUT_DIR / filename.replace('.csv', '_processed.csv')
    df_proc.to_csv(out_path, index=False)
    print(f'  Gespeichert: {out_path.name}')

print(f'\n✅ Alle Dateien gespeichert in: {OUTPUT_DIR}')

---
## Hinweise zur Erweiterung

| Schritt | Erweiterung |
|---------|-------------|
| Sampling-Rate | Für S05–S11 (2000 Hz): `FS_ORIGINAL = 2000` setzen → Resampling wird automatisch übersprungen |
| MVC-Normalisierung | Nach Schritt 5: `df_norm[col] = df_norm[col] / mvc_value * 100` |
| Mehrere Probanden | Äußere Schleife über Probanden-Ordner, `FS_ORIGINAL` je nach Proband setzen |
| Speicherformat | Statt CSV auch `.npy` oder `.pkl` für schnellere I/O-Performance |
